# 데이터셋별 LoRA SFT 효과 측정 — Colab Pro (L4)

`data_ablation/README.md`의 실험을 Colab에서 돌리는 래퍼입니다. 코드는 GitHub `data-ablation` 브랜치에서 받고, 결과(`runs/`)는 Google Drive에 두어 세션이 끊겨도 남습니다.

**사용법**
1. 런타임 → 런타임 유형 변경 → **L4 GPU** (A100도 됨, CU 4배).
2. 아래 설정 셀에서 `COND`를 고르고 **셀을 위에서부터 차례로 실행**합니다.
3. 세션이 끊기면 다시 연결한 뒤 **같은 설정으로 1~4번 셀을 다시 실행**합니다. 끝난 단계는 건너뛰고, self-CoT·학습은 저장된 지점부터, 평가는 끝난 seed 다음부터 이어집니다.
4. 한 조건이 끝나면 5번 셀로 표를 만들고, 6번 셀로 결과를 GitHub `results/<RUN_NAME>` 브랜치에 올립니다(선택).

**순서 권장**: `base`(L4에서 베이스라인 seed 0·1 재측정, 약 6시간) → `cosyn` → `vwi` → `pmcvqa`. `base` 결과가 있으면 그것을 비교 기준으로 씁니다(같은 GPU). 없으면 main의 `results/seed*`(RTX 4090)와 비교합니다.

**시간 (L4)**: base 약 6h · cosyn 약 13h · vwi 약 15h · pmcvqa 약 18h. 세션당 2~4시간씩 끊어 돌리면 됩니다. 브라우저 탭은 열어 두세요(Pro는 백그라운드 실행이 없음).

In [ ]:
#@title 1. 설정
COND = "cosyn"            #@param ["base", "cosyn", "vwi", "pmcvqa"]
SEEDS = "0 1"             #@param {type:"string"}
RUN_NAME = ""             #@param {type:"string"}   # 비우면 abl_<COND> 또는 base_l4
BRANCH = "dongjun-lee"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MMDL-MMMU"  #@param {type:"string"}
STEPS = 1500              #@param {type:"integer"}
BS = 2                    #@param {type:"integer"}  # L4 22.5GB: 2 x 4. A100 80GB: 8 x 1
ACCUM = 4                 #@param {type:"integer"}
SMOKE = False             #@param {type:"boolean"}  # True: 200 레코드·20 step·30문항 평가로 전 경로만 점검 (약 40분)

import os
RUN_NAME = RUN_NAME or ("base_l4" if COND == "base" else f"abl_{COND}")
ENV = dict(RUN_NAME=RUN_NAME, SEEDS=SEEDS, STEPS=str(STEPS), BS=str(BS), ACCUM=str(ACCUM),
           FAST_DIR="/content/fast", MERGED_ROOT="/content/merged", HF_HOME="/content/hf",
           VLLM_BATCH_INVARIANT="1", TOKENIZERS_PARALLELISM="false")
if SMOKE:
    ENV.update(SMOKE="1", N_RECORDS="200", MIN_RECORDS="50", STEPS="20", RUN_NAME=f"smoke_{COND}")
os.environ.update(ENV)
os.environ["DRIVE_ROOT"], os.environ["BRANCH"], os.environ["COND"] = DRIVE_ROOT, BRANCH, COND
print({k: os.environ[k] for k in ENV})

In [ ]:
#@title 2. Drive 마운트 + 코드 받기 (runs/ -> Drive)
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
%cd /content
if not os.path.isdir("MMDL-MMMU"):
    !git clone -q -b $BRANCH https://github.com/jinyoung924/MMDL-MMMU.git
%cd /content/MMDL-MMMU
!git fetch -q origin && git checkout -q $BRANCH && git pull -q --ff-only origin $BRANCH && git log --oneline -1
!rm -rf runs && ln -s $DRIVE_ROOT/runs runs && ls -la runs/
# 선행 base_l4 결과가 Drive에 있으면 그것을 비교 기준으로 (같은 GPU), 없으면 main의 results/seed* (RTX 4090)
base = [f"runs/base_l4/eval/seed{s}" for s in SEEDS.split() if os.path.exists(f"runs/base_l4/eval/seed{s}/scores.json")]
os.environ["BASE_DIRS"] = " ".join(base) if len(base) == len(SEEDS.split()) else "results/seed0 results/seed1 results/seed2 results/seed3 results/seed4"
print("BASE_DIRS =", os.environ["BASE_DIRS"])

In [ ]:
#@title 3. 환경 설치 (세션마다 약 10분; venv는 로컬 디스크)
!bash data_ablation/setup_colab.sh

In [ ]:
#@title 4. 실행 (끊기면 이 셀만 다시 실행)
%cd /content/MMDL-MMMU
if COND == "base":
    !source /content/venv/bin/activate && bash data_ablation/run_base.sh
else:
    !source /content/venv/bin/activate && bash data_ablation/run_condition.sh $COND

In [ ]:
#@title 5. 조건별 요약표 (Drive의 runs/abl_* 전부)
%cd /content/MMDL-MMMU
!source /content/venv/bin/activate && python data_ablation/summarize.py --runs $(ls -d runs/abl_* 2>/dev/null) --base $BASE_DIRS --out runs/results_colab.md
print(open("runs/results_colab.md").read())

In [ ]:
#@title 6. (선택) 결과를 GitHub results/<RUN_NAME> 브랜치로 push
# Colab 좌측 🔑 보안 비밀에 GITHUB_TOKEN(이 레포 Contents: read/write) 저장. 가중치·이미지·jsonl은 .gitignore로 제외됨.
from google.colab import userdata
import os, subprocess
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
%cd /content/MMDL-MMMU
!git checkout -q -B results/$RUN_NAME && rm -f runs && mkdir -p runs && rsync -a --exclude images --exclude '*.jsonl' --exclude 'step*' --exclude '*_merged' --exclude '*.tmp' $DRIVE_ROOT/runs/$RUN_NAME runs/
!bash cloud/push_results.sh runs/$RUN_NAME
!git checkout -q $BRANCH && rm -rf runs && ln -s $DRIVE_ROOT/runs runs && echo restored